# Hazard curves (credit survival and default intensity)

Deep-dive reference: **hazard rate curves** for credit: survival probabilities and instantaneous default intensity.

## Concept

A **hazard curve** describes the risk of default through an **instantaneous hazard rate** \(\lambda(t)\) (and often a **recovery rate**). **Survival** to time \(t\) is \(S(t) = \exp(-\int_0^t \lambda(u)\,du)\) under standard assumptions. Market quotes are often **CDS spreads** or **survival** at pillar times; in this API you supply **\((t, \lambda)\) knots** and read off **`survival(t)`** and **`hazard_rate(t)`**.

## API walkthrough

If `HazardCurve` is available, knots are **`(time_years, hazard_rate)`**. Use **`survival(t)`** for no-default probability and **`hazard_rate(t)`** for the intensity at `t`.

In [1]:
try:
    from datetime import date

    from finstack.core.market_data import HazardCurve
except ImportError as exc:
    print("HazardCurve is not importable from finstack.core.market_data:", exc)
else:
    base = date(2024, 1, 1)
    # Investment-grade style: lower hazard
    ig = HazardCurve(
        "IG-CREDIT",
        base,
        [(0.0, 0.008), (1.0, 0.01), (3.0, 0.012), (5.0, 0.014)],
        recovery_rate=0.4,
    )
    # High-yield style: higher hazard
    hy = HazardCurve(
        "HY-CREDIT",
        base,
        [(0.0, 0.04), (1.0, 0.045), (3.0, 0.05), (5.0, 0.055)],
        recovery_rate=0.35,
    )
    print("IG curve:", ig)
    print("HY curve:", hy)
    for name, hc in [("IG", ig), ("HY", hy)]:
        print(f"\n{name} survival / hazard at selected tenors:")
        for t in (0.5, 1.0, 3.0, 5.0):
            print(
                f"  t={t}y  survival={hc.survival(t):.6f}  hazard_rate={hc.hazard_rate(t):.6f}",
            )

IG curve: HazardCurve(id="IG-CREDIT")
HY curve: HazardCurve(id="HY-CREDIT")

IG survival / hazard at selected tenors:
  t=0.5y  survival=0.995012  hazard_rate=0.010000
  t=1.0y  survival=0.990050  hazard_rate=0.010000
  t=3.0y  survival=0.966572  hazard_rate=0.012000
  t=5.0y  survival=0.939883  hazard_rate=0.014000

HY survival / hazard at selected tenors:
  t=0.5y  survival=0.977751  hazard_rate=0.045000
  t=1.0y  survival=0.955997  hazard_rate=0.045000
  t=3.0y  survival=0.865022  hazard_rate=0.050000
  t=5.0y  survival=0.774916  hazard_rate=0.055000


## Practical example

Compare **5Y survival** between two stylized credits (IG vs HY) for a quick relative-risk snapshot.

In [2]:
try:
    from datetime import date

    from finstack.core.market_data import HazardCurve
except ImportError as exc:
    print("Skip practical example — HazardCurve unavailable:", exc)
else:
    base = date(2024, 1, 1)
    ig = HazardCurve("IG", base, [(0.0, 0.008), (5.0, 0.014)], recovery_rate=0.4)
    hy = HazardCurve("HY", base, [(0.0, 0.045), (5.0, 0.055)], recovery_rate=0.35)
    t = 5.0
    s_ig, s_hy = ig.survival(t), hy.survival(t)
    print(f"5Y survival IG = {s_ig:.6f}")
    print(f"5Y survival HY = {s_hy:.6f}")
    print(f"HY cumulative default prob ~ {1.0 - s_hy:.6f} (stylized, no market calibration)")

5Y survival IG = 0.932394
5Y survival HY = 0.759572
HY cumulative default prob ~ 0.240428 (stylized, no market calibration)


## Takeaways

- **`HazardCurve`** is built from **hazard rate pillars**; **`survival(t)`** is the primary risk metric for pricing and limits.
- **`recovery_rate`** is part of the curve object and matters for **CDS / bond** recovery assumptions.
- **IG vs HY** is visible in both **level** and **term shape** of \(\lambda\); real desks calibrate from **CDS quotes**, not hand-waved constants.

## Calibration from CDS par spreads

Real hazard curves are **bootstrapped from CDS quotes**, not built from guessed hazard rate knots.  The calibration engine strips survival probabilities from observed par spreads, using a discount curve for present-value calculations.

Below we run a **two-step plan**: first bootstrap a discount curve, then calibrate the hazard curve from CDS par spreads referencing that discount curve.

In [3]:
import json

from finstack.valuations import calibrate

def tenor(count, unit):
    return {"tenor": {"count": count, "unit": unit}}

plan = {
    "schema": "finstack.calibration/2",
    "plan": {
        "id": "aapl-hazard",
        "description": "Discount curve + AAPL hazard curve from CDS par spreads",
        "quote_sets": {
            "ois": [
                {"class": "rates", "type": "deposit", "id": "DEP-3M", "index": "USD-SOFR", "pillar": tenor(3, "months"), "rate": 0.0530},
                {"class": "rates", "type": "swap", "id": "SWP-1Y", "index": "USD-SOFR-OIS", "pillar": tenor(1, "years"), "rate": 0.0540},
                {"class": "rates", "type": "swap", "id": "SWP-5Y", "index": "USD-SOFR-OIS", "pillar": tenor(5, "years"), "rate": 0.0420},
                {"class": "rates", "type": "swap", "id": "SWP-10Y", "index": "USD-SOFR-OIS", "pillar": tenor(10, "years"), "rate": 0.0410},
            ],
            "cds": [
                {"class": "cds", "type": "cds_par_spread", "id": "AAPL-1Y", "entity": "AAPL",
                 "pillar": tenor(1, "years"), "spread_bp": 25.0, "recovery_rate": 0.4,
                 "convention": {"currency": "USD", "doc_clause": "isda_na"}},
                {"class": "cds", "type": "cds_par_spread", "id": "AAPL-3Y", "entity": "AAPL",
                 "pillar": tenor(3, "years"), "spread_bp": 35.0, "recovery_rate": 0.4,
                 "convention": {"currency": "USD", "doc_clause": "isda_na"}},
                {"class": "cds", "type": "cds_par_spread", "id": "AAPL-5Y", "entity": "AAPL",
                 "pillar": tenor(5, "years"), "spread_bp": 45.0, "recovery_rate": 0.4,
                 "convention": {"currency": "USD", "doc_clause": "isda_na"}},
                {"class": "cds", "type": "cds_par_spread", "id": "AAPL-7Y", "entity": "AAPL",
                 "pillar": tenor(7, "years"), "spread_bp": 55.0, "recovery_rate": 0.4,
                 "convention": {"currency": "USD", "doc_clause": "isda_na"}},
                {"class": "cds", "type": "cds_par_spread", "id": "AAPL-10Y", "entity": "AAPL",
                 "pillar": tenor(10, "years"), "spread_bp": 65.0, "recovery_rate": 0.4,
                 "convention": {"currency": "USD", "doc_clause": "isda_na"}},
            ],
        },
        "steps": [
            {
                "id": "USD-OIS",
                "quote_set": "ois",
                "kind": "discount",
                "curve_id": "USD-OIS",
                "currency": "USD",
                "base_date": "2025-01-15",
                "method": "Bootstrap",
                "interpolation": "linear",
                "extrapolation": "flat_forward",
            },
            {
                "id": "AAPL-HAZARD",
                "quote_set": "cds",
                "kind": "hazard",
                "curve_id": "AAPL-SR-USD",
                "entity": "AAPL",
                "seniority": "senior",
                "currency": "USD",
                "base_date": "2025-01-15",
                "discount_curve_id": "USD-OIS",
                "recovery_rate": 0.4,
                "method": "Bootstrap",
                "interpolation": "linear",
            },
        ],
        "settings": {"use_parallel": False},
    },
}

result = calibrate(json.dumps(plan))
print("Success:", result.success)
print(result.report_to_dataframe().to_string(index=False))
print()

hzd = result.market.get_hazard("AAPL-SR-USD")
print("Calibrated AAPL hazard curve (survival and hazard rates):")
for t in [0.5, 1.0, 3.0, 5.0, 7.0, 10.0]:
    print(f"  t={t:5.1f}y  survival={hzd.survival(t):.6f}  hazard_rate={hzd.hazard_rate(t):.6f}")

Success: True
    step_id  success  iterations  max_residual         rmse                                                                          convergence_reason
AAPL-HAZARD     True         402  5.476869e-13 3.872686e-13 generic bootstrap calibration converged: max_residual (5.48e-13) within tolerance (1.00e-8)
    USD-OIS     True         317  9.259093e-13 7.097110e-13 generic bootstrap calibration converged: max_residual (9.26e-13) within tolerance (1.00e-8)

Calibrated AAPL hazard curve (survival and hazard rates):
  t=  0.5y  survival=0.997768  hazard_rate=0.004468
  t=  1.0y  survival=0.995412  hazard_rate=0.006702
  t=  3.0y  survival=0.982051  hazard_rate=0.010269
  t=  5.0y  survival=0.962086  hazard_rate=0.010269
  t=  7.0y  survival=0.935379  hazard_rate=0.014076
  t= 10.0y  survival=0.892014  hazard_rate=0.015840
